In [176]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [177]:
class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad=0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label=label


  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0*out.grad
      other.grad += 1.0*out.grad
    out._backward = _backward

    return out

  def __mul__(self, other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data*out.grad
      other.grad += self.data*out.grad
    out._backward = _backward
    return out

  def pow(self,other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')
    def _backward():
      self.grad += other*(self.data**(other-1))*out.grad
    out._backward = _backward
    return out

  def __rmul__(self, other):
    return self * other

  def __truediv__(self, other):
    other = other if isinstance(other, Value) else Value(other) # Ensure other is a Value object
    return self * other.pow(-1.0) # Correctly implement division using multiplication and power

  def __sub__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data - other.data, (self, other), '-')
    # Backward pass for subtraction is similar to addition
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad -= 1.0 * out.grad # Negative gradient for the subtrahend
    out._backward = _backward
    return out


  def tanh(self):
    n = self.data
    t=Value((math.exp(2*n)-1)/(math.exp(2*n)+1))
    out=Value(t.data,(self,),'tanh')
    def _backward():
      self.grad += (1-t.data**2)*out.grad
    out._backward = _backward
    return out

  def exp(self):
    x=self.data
    out=Value(math.exp(x),(self,),'exp')
    def _backward():
      self.grad += out.data*out.grad
    out._backward = _backward
    return out

  def backward(self):
    topo=[]
    visited=set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad=1.0
    for node in reversed(topo):
      node._backward()


a = Value(2.0,label='a')
b = Value(-3.0,label='b')
c = Value(10.0,label='c')
e=a*b;e.label='e'
d=e+c;d.label='d'
f=Value(-2.0,label='f')
L=d*f;L.label='L'
L

Value(data=-8.0)

In [178]:
#inputs x1,x2
x1 = Value(2.0,label='x1')
x2 = Value(0.0,label='x2')
#weights w1,w2
w1 = Value(-3.0,label='w1')
w2 = Value(1.0,label='w2')
#bias
b = Value(6.8813735870195432,label='b')

x1w1=x1*w1;x1w1.label='x1w1'
x2w2=x2*w2;x2w2.label='x2w2'
x1w1x2w2=x1w1+x2w2;x1w1x2w2.label='x1w1+x2w2'
n=x1w1x2w2+b;n.label='n'

e=((2*n).exp())
o=(e-1)/(e+1)

o.backward()

In [179]:
class Neuron:

  def __init__(self, nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))

  def __call__(self,x):
    act=__builtins__.sum((wi*xi for wi,xi in zip(self.w,x)), self.b)
    out=act.tanh()
    return out

  def parameters(self):
    return self.w+[self.b]

class Layer:
  def __init__(self,nin,nout):
    self.neurons=[Neuron(nin) for _ in range(nout)]

  def __call__(self,x):
    outs=[n(x) for n in self.neurons]
    return outs

  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
  def __init__(self,nin,nouts):
    sz=[nin]+nouts
    self.layers=[Layer(sz[i],sz[i+1]) for i in range(len(nouts))]

  def __call__(self,x):
    for layer in self.layers:
      x=layer(x)
    return x
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

x=[2.0,3.0,-1.0]
n = MLP(3,[4,4,1])
n(x)

[Value(data=0.8398599350618873)]

In [180]:
n.parameters()

[Value(data=-0.9665581781871955),
 Value(data=0.5526621330707879),
 Value(data=0.13020965939899165),
 Value(data=0.661236132776009),
 Value(data=-0.754042282485655),
 Value(data=-0.03359442442743932),
 Value(data=0.5488371838623196),
 Value(data=-0.7838733323066478),
 Value(data=-0.27290091281435513),
 Value(data=-0.5147581458062804),
 Value(data=0.5182939510447087),
 Value(data=0.48010231703569217),
 Value(data=0.689287107008933),
 Value(data=-0.25332794043994866),
 Value(data=0.9947081626422098),
 Value(data=-0.22746102542896218),
 Value(data=-0.3992555773104478),
 Value(data=-0.4276172457872587),
 Value(data=0.3630567733082566),
 Value(data=-0.7152911058211717),
 Value(data=-0.9788638928774371),
 Value(data=0.3966937982992855),
 Value(data=-0.5133434416900022),
 Value(data=-0.44614166856334325),
 Value(data=-0.464169461670737),
 Value(data=0.4249486227227126),
 Value(data=-0.533419027938655),
 Value(data=0.45060009072738283),
 Value(data=0.0034507516066282218),
 Value(data=0.1948306

In [181]:
xs = [
    [2.0,3.0,-1.0],
    [3.0,-1.0,0.5],
    [0.5,1.0,1.0],
    [1.0,1.0,-1.0]
]

ys=[1.0,-1.0,-1.0,1.0]#desired targets

In [205]:
for k in range (20):
  ypred=[n(x) for x in xs]
  loss = sum(((yout[0]-ygt).pow(2) for ygt,yout in zip(ys,ypred)), start=Value(0.0))
  for p in n.parameters():
    p.grad=0.0
  loss.backward()
  for p in n.parameters():
    p.data = p.data + (-0.05)*p.grad

  print(k,loss.data)


0 0.0722132642971072
1 0.0019073739608126425
2 0.0003044192955469876
3 0.00027732823150726054
4 0.0010261114703738773
5 0.0141117588086055
6 0.29998982835814514
7 6.289587756130144e-06
8 9.415799003396235e-09
9 1.9289280203973312e-10
10 1.529833174377677e-11
11 1.9500858958328005e-12
12 3.0023388551310076e-13
13 5.047598946157006e-14
14 8.806231752367104e-15
15 1.54920450419592e-15
16 2.7155122224779863e-16
17 4.7418593783186204e-17
18 8.294435015854179e-18
19 1.4623934271071309e-18


In [206]:
ypred

[[Value(data=0.9999999997985772)],
 [Value(data=-0.9999999990393885)],
 [Value(data=-0.9999999993228896)],
 [Value(data=0.9999999997985815)]]